In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import inspect
import openpyxl
from joblib import Parallel, delayed
import shutil
import gc
import logging
import sys
sys.path.insert(0, r"C:\Users\meny\OneDrive - Region Sjælland\AX3\actimotus")
sys.path.insert(0, r"C:\Users\meny\OneDrive - Region Sjælland\AX3\actimotus\src")
from analysis.activity_pipeline import load_raw_file, process_file


# Skru ned globalt
logging.basicConfig(level=logging.WARNING, force=True)

# Definer stier
file_path = Path(r"C:\Users\meny\Downloads\test")
file_path_output = Path(r"C:\Users\meny\Downloads\test\results")
file_path_output.mkdir(parents=True, exist_ok=True)

# Mappe til per-fil exposures CSV
exposures_dir = file_path_output / "exposures_csv"
exposures_dir.mkdir(parents=True, exist_ok=True)

# i hbsc - skal akserne roteres sådan her: Efter nærmere eftertanke, så tror jeg det er forkert. Det burde være x=y, y=-x, z=z
# df['acc_y'] = df['z'] * (-1)
# df['acc_x'] = df['y'] 
# df['acc_z'] = df['x'] * (-1)




# Definér rotations-funktion
def rotate_axes(df):
    # rotere akserne, så de stemmer overens med standard
    df['acc_y'] = df['y'] *(-1)
    df['acc_z'] = df['z'] 
    df['acc_x'] = df['x'] *(-1)


# Gem rotationskoden som tekst (til metadata-sheet)
axis_rotation_code = inspect.getsource(rotate_axes)

# Tilføj efter axis_rotation_code definition
error_log = file_path_output / "error_log.txt"


  
all_cwa_files = list(file_path.glob("**/*.cwa"))

unique_cwa_files = {}
duplicate_names = []

for cwa_file in all_cwa_files:
    if cwa_file.name in unique_cwa_files:
        duplicate_names.append(cwa_file.name)
    else:
        unique_cwa_files[cwa_file.name] = cwa_file

cwa_files = list(unique_cwa_files.values())

print(f"Finder {len(all_cwa_files)} filer i alt")
print(f"Beholder {len(cwa_files)} unikke filnavne")

if duplicate_names:
    print("Dubletter fundet for disse filnavne:")
    print(sorted(set(duplicate_names)))    
        
    


all_done = []
for cwa_file in cwa_files:
    df = load_raw_file(cwa_file)
    rotate_axes(df)
    df_subset = df[['acc_x', 'acc_y', 'acc_z']].copy()
    del df; gc.collect()
    all_done.append(process_file(df_subset, cwa_file.stem, file_path_output, exposures_dir, axis_rotation_code, error_log))
    del df_subset; gc.collect()

# tilføj log:
successful = sum(1 for r in all_done if r and r[1] == "SUCCESS")
failed = sum(1 for r in all_done if r and r[1] == "FAILED")

print(f"\n{'='*60}")
print(f"✓ Successful: {successful}/{len(cwa_files)}")
print(f"✗ Failed: {failed}/{len(cwa_files)}")
if failed > 0:
    print(f"Se fejl i: {error_log}")




# --- Stream-merge alle per-fil CSV til én samlet CSV uden at loade i RAM ---
combined_csv = file_path_output / "all_exposures.csv"
csv_files = sorted(exposures_dir.glob("*.csv"))

with open(combined_csv, "w", encoding="utf-8", newline="") as out_f:
    first = True
    for f in csv_files:
        with open(f, "r", encoding="utf-8") as in_f:
            if first:
                shutil.copyfileobj(in_f, out_f)  # inkl. header
                first = False
            else:
                next(in_f)  # skip header
                shutil.copyfileobj(in_f, out_f)

print(f"\nFærdig! Samlet exposures gemt i: {combined_csv}")

ModuleNotFoundError: No module named 'acti_motus'

In [ ]:
korrekt  df['acc_y'] = df['z'] * (-1)    
    df['acc_x'] = df['y'] 
    df['acc_z'] = df['x'] * (-1)